# End-to-End Usage Example (BERT-pivot)

This notebook demonstrates the full workflow for `intervene_enc` with the
bidirectional encoder + per-outcome (risk, time) task heads.

* **Phase 1** trains the embedder unchanged from the AR pipeline.
* **Phase 2** pre-trains the encoder with masked language modelling
  (atomic-interval mask + two time-aware auxiliaries).
* **Phase 3** attaches `TaskHeads` and fine-tunes for per-patient
  outcome risk + time-to-event regression.

Evaluation is a single forward pass: no autoregressive trajectory generation.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free,compute_cap --format=csv,noheader


In [ ]:
%pip install -e .


In [ ]:
import importlib
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import intervene_enc.config.dataset_config as dataset_config
import intervene_enc.config.model_config as model_config
from intervene_enc import dataset, schedulers, utils, loss, embedder, transformer, inference

# Preserve runtime keys (ctx_dim is set after we see the data; not in source).
_old_model_config = dict(getattr(model_config, "MODEL_CONFIG", {}))
_old_training_settings = dict(getattr(model_config, "TRAINING_SETTINGS", {}))

for module in (dataset_config, model_config, dataset, schedulers, utils,
               loss, embedder, transformer, inference):
    importlib.reload(module)

for k, v in _old_model_config.items():
    model_config.MODEL_CONFIG.setdefault(k, v)
for k, v in _old_training_settings.items():
    model_config.TRAINING_SETTINGS.setdefault(k, v)

from intervene_enc.config.model_config import MODEL_CONFIG, TRAINING_SETTINGS

print(f"ctx_dim = {MODEL_CONFIG.get('ctx_dim')}  |  embed_dim = {MODEL_CONFIG.get('embed_dim')}")


In [ ]:
# Optional cleanup controls for checkpoint artifacts.
CLEAN_SCALER = False
CLEAN_TOKENIZER = False
CLEAN_EMBEDDER = False
CLEAN_TRANSFORMER = False
CLEAN_ALL = True

clean_scaler      = CLEAN_SCALER or CLEAN_ALL
clean_tokenizer   = CLEAN_TOKENIZER or CLEAN_ALL
clean_embedder    = CLEAN_EMBEDDER or CLEAN_ALL
clean_transformer = CLEAN_TRANSFORMER or CLEAN_ALL

paths_to_delete = []
if clean_scaler:
    paths_to_delete.append(Path(model_config.CHECKPOINT_PATH) / "scaler.pkl")
if clean_tokenizer:
    paths_to_delete.append(Path(model_config.CHECKPOINT_PATH) / "tokenizer.pt")
if clean_embedder:
    p1 = Path(model_config.PHASE1_CHECKPOINT).resolve()
    paths_to_delete += [p1, p1.parent / "ckpt_last.pt"]
if clean_transformer:
    for ph in (model_config.PHASE2_CHECKPOINT, model_config.PHASE3_CHECKPOINT):
        ck = Path(ph).resolve()
        paths_to_delete += [ck, ck.parent / "ckpt_last.pt"]

removed, missing = [], []
for path in dict.fromkeys(paths_to_delete):
    if path.exists():
        try:
            path.unlink()
            removed.append(path)
        except Exception as e:
            print(f"  skip {path}: {e}")
    else:
        missing.append(path)
print(f"Cleanup: removed {len(removed)}, missing {len(missing)}")


## 1) Data Load and Processing


In [ ]:
# Pre-split data load (data/train/, data/test/ produced by generate_synthetic_data.ipynb).
train_temporal_raw = pd.read_csv(dataset_config.TRAIN_TEMPORAL_DATA_FILE, low_memory=False)
train_ctx_raw      = pd.read_csv(dataset_config.TRAIN_CTX_DATA_FILE)
eval_temporal_raw  = pd.read_csv(dataset_config.TEST_TEMPORAL_DATA_FILE,  low_memory=False)
eval_ctx_raw       = pd.read_csv(dataset_config.TEST_CTX_DATA_FILE)

# Train / val split (80/20 stratified is overkill on synthetic data; plain split is fine).
TRAIN_VAL_SIZE       = 0.2
TRAIN_VAL_SPLIT_SEED = 42

NUM_WORKERS = 0  # Jupyter / Windows safe default

# Process the train pool first so the scaler is fit + saved.
print("Processing full train pool (fits scaler)...")
train_pool_processor = dataset.DataProcessor(
    train_temporal_raw.copy(), train_ctx_raw.copy(),
    scaler=None,
    tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
)
train_pool_df, train_pool_ctx_df = train_pool_processor.run()
scaler = joblib.load(Path(model_config.CHECKPOINT_PATH) / "scaler.pkl")

# Tokenizer.
TOKENIZER_PATH = Path(model_config.CHECKPOINT_PATH) / "tokenizer.pt"
if TOKENIZER_PATH.exists():
    tokenizer = dataset.EMRTokenizer.load(TOKENIZER_PATH)
else:
    tokenizer = dataset.EMRTokenizer.from_processed_df(train_pool_df)
    tokenizer.save(TOKENIZER_PATH)

train_pool_pids = train_pool_df["PatientId"].dropna().unique()
train_ids, val_ids = train_test_split(
    train_pool_pids, test_size=TRAIN_VAL_SIZE, random_state=TRAIN_VAL_SPLIT_SEED,
)

train_temporal_df = train_pool_df[train_pool_df["PatientId"].isin(train_ids)].copy()
train_ctx_df      = train_pool_ctx_df[train_pool_ctx_df.index.isin(train_ids)].copy()

val_temporal_raw  = train_temporal_raw[train_temporal_raw["PatientId"].isin(val_ids)].copy()
val_ctx_raw       = train_ctx_raw[train_ctx_raw["PatientId"].isin(val_ids)].copy()
val_temporal_df, val_ctx_df = dataset.DataProcessor(
    val_temporal_raw, val_ctx_raw,
    scaler=scaler, tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
).run()

eval_temporal_df, eval_ctx_df = dataset.DataProcessor(
    eval_temporal_raw.copy(), eval_ctx_raw.copy(),
    scaler=scaler, tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
).run()

train_ds = dataset.EMRDataset(train_temporal_df, train_ctx_df, tokenizer=tokenizer)
val_ds   = dataset.EMRDataset(val_temporal_df,   val_ctx_df,   tokenizer=tokenizer)
eval_ds  = dataset.EMRDataset(eval_temporal_df,  eval_ctx_df,  tokenizer=tokenizer)

# Auto-detect ctx_dim from the materialised context dataframe.
model_config.MODEL_CONFIG["ctx_dim"] = int(train_ds.context_df.shape[1])
print(f"ctx_dim -> {model_config.MODEL_CONFIG['ctx_dim']}")
print(f"Train patients: {len(train_ids)} | Val: {len(val_ids)} | Eval: {eval_temporal_df['PatientId'].nunique()}")


## 2) Training


In [ ]:
# Phase control flags — flip to False to skip a phase and load its checkpoint instead.
RUN_PHASE1      = True
RUN_PHASE2      = True
RUN_PHASE3      = True
RESUME_TRAINING = True
print(f"RUN_PHASE1={RUN_PHASE1}  RUN_PHASE2={RUN_PHASE2}  RUN_PHASE3={RUN_PHASE3}  RESUME={RESUME_TRAINING}")


In [ ]:
# DataLoaders — all three phases use the natural distribution.
bs = model_config.TRAINING_SETTINGS["batch_size"]
train_dl = dataset.get_dataloader(train_ds, batch_size=bs, collate_fn=dataset.collate_emr,
                                  oversample=False, bucket_batching=True, num_workers=NUM_WORKERS)
val_dl   = dataset.get_dataloader(val_ds,   batch_size=bs, collate_fn=dataset.collate_emr,
                                  oversample=False, bucket_batching=True, num_workers=NUM_WORKERS)
print(f"DataLoaders: train={len(train_dl)} batches  val={len(val_dl)} batches  (bs={bs})")


In [ ]:
def _load_embedder_checkpoint(best_path, tokenizer):
    ckpt_best = Path(best_path).resolve()
    ckpt_last = ckpt_best.parent / "ckpt_last.pt"
    ckpt = ckpt_best if ckpt_best.exists() else ckpt_last
    if not ckpt.exists():
        raise FileNotFoundError("No Phase-1 checkpoint found. Set RUN_PHASE1=True first.")
    model, *_ = embedder.EMREmbedding.load(ckpt, tokenizer=tokenizer)
    return model


def _load_encoder_checkpoint(best_path, embedder_model, attach_task_heads=None):
    ckpt_best = Path(best_path).resolve()
    ckpt_last = ckpt_best.parent / "ckpt_last.pt"
    ckpt = ckpt_best if ckpt_best.exists() else ckpt_last
    if not ckpt.exists():
        raise FileNotFoundError(f"No checkpoint at {best_path}. Train the relevant phase first.")
    model, *_ = transformer.InterveneEncoder.load(ckpt, embedder=embedder_model,
                                            attach_task_heads=attach_task_heads)
    return model


# ----- Phase 1: embedder -----
if RUN_PHASE1:
    embedder_model = embedder.EMREmbedding(
        tokenizer=tokenizer,
        ctx_dim=model_config.MODEL_CONFIG["ctx_dim"],
        time2vec_dim=model_config.MODEL_CONFIG["time2vec_dim"],
        embed_dim=model_config.MODEL_CONFIG["embed_dim"],
    )
    embedder_model, _, _ = embedder.train_embedder(
        embedder=embedder_model,
        train_loader=train_dl, val_loader=val_dl,
        resume=RESUME_TRAINING,
        checkpoint_path=model_config.PHASE1_CHECKPOINT,
        training_settings=model_config.TRAINING_SETTINGS,
    )
else:
    embedder_model = _load_embedder_checkpoint(model_config.PHASE1_CHECKPOINT, tokenizer)

# ----- Phase 2: bidirectional MLM pre-training -----
if RUN_PHASE2:
    encoder = transformer.InterveneEncoder(cfg=model_config.MODEL_CONFIG, embedder=embedder_model)
    encoder, _, _ = transformer.pretrain_transformer(
        model=encoder, train_dl=train_dl, val_dl=val_dl,
        resume=RESUME_TRAINING, checkpoint_path=model_config.PHASE2_CHECKPOINT,
        training_settings=model_config.TRAINING_SETTINGS,
    )
else:
    encoder = _load_encoder_checkpoint(model_config.PHASE2_CHECKPOINT, embedder_model,
                                       attach_task_heads=False)

# ----- Phase 3: outcome + time fine-tune -----
if RUN_PHASE3:
    if encoder.task_heads is None:
        encoder.attach_task_heads()
    encoder, _, _ = transformer.finetune_transformer(
        model=encoder, train_dl=train_dl, val_dl=val_dl,
        resume=RESUME_TRAINING, checkpoint_path=model_config.PHASE3_CHECKPOINT,
        training_settings=model_config.TRAINING_SETTINGS,
    )
else:
    encoder = _load_encoder_checkpoint(model_config.PHASE3_CHECKPOINT, embedder_model,
                                       attach_task_heads=True)

encoder.eval()
print("Model ready for evaluation.")


## 3) Inference — single bidirectional pass

The encoder consumes the same `EMRDataset` as the AR pipeline but does **not**
generate a trajectory.  `inference.predict` returns a one-row-per-patient
dataframe with `P_<outcome>` (sigmoid of the risk-head logit) and
`T_<outcome>` (softplus of the time-head output, hours).


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

# Truncate test sequences to the same window the model is calibrated on.
EVAL_INPUT_DAYS = 2
print(f"Building truncated eval dataset ({EVAL_INPUT_DAYS}-day input window)...")
eval_input_processor = dataset.DataProcessor(
    eval_temporal_raw.copy(), eval_ctx_raw.copy(),
    scaler=scaler, tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
    max_input_days=EVAL_INPUT_DAYS,
)
eval_input_temporal_df, eval_input_ctx_df = eval_input_processor.run()
eval_ds_input = dataset.EMRDataset(eval_input_temporal_df, eval_input_ctx_df,
                                   tokenizer=tokenizer)

predictions = inference.predict(encoder, eval_ds_input,
                                batch_size=model_config.TRAINING_SETTINGS["batch_size"])
print(f"Predictions: {predictions.shape[0]} patients  cols={list(predictions.columns)[:6]} ...")
predictions.head()


### 3a) Ground-truth extraction


In [ ]:
from intervene_enc.config.dataset_config import OUTCOME_RARE_THRESHOLD_PCT, RELEASE_TOKEN

EVAL_FULL_HORIZON_HOURS = 336.0
EVAL_PREVALENCE_THRESHOLD = OUTCOME_RARE_THRESHOLD_PCT / 100.0
AUC_EXCLUDE = (RELEASE_TOKEN,)  # P(release) is just 1 - P(death) here; report separately.

OUTCOME_NAMES = encoder.outcome_names


def extract_ground_truth(eval_ds, outcome_names):
    """Per-patient first-occurrence times from the un-truncated test sequences."""
    outcome_set = set(outcome_names)
    gt = {}
    for pid in eval_ds.patient_ids:
        df = eval_ds.patient_groups[pid]
        patient_gt = {n: np.inf for n in outcome_names}
        for _, row in df.iterrows():
            tok = row["PositionToken"]
            if tok in outcome_set:
                t = row["TimePoint"]
                if t < patient_gt[tok]:
                    patient_gt[tok] = t
        gt[pid] = patient_gt
    return gt


def extract_ground_truth_episodes(eval_ds, outcome_names):
    outcome_set = set(outcome_names)
    gt = {}
    for pid in eval_ds.patient_ids:
        df = eval_ds.patient_groups[pid]
        patient_gt = {n: [] for n in outcome_names}
        for _, row in df.iterrows():
            tok = row["PositionToken"]
            if tok in outcome_set:
                patient_gt[tok].append(row["TimePoint"])
        gt[pid] = patient_gt
    return gt


gt_first    = extract_ground_truth(eval_ds, OUTCOME_NAMES)
gt_episodes = extract_ground_truth_episodes(eval_ds, OUTCOME_NAMES)
print(f"GT extracted for {len(gt_first)} patients.")

summary = {n: sum(1 for p in gt_first if gt_first[p][n] < np.inf) for n in OUTCOME_NAMES}
pd.Series(summary, name="patients_with_complication").sort_values(ascending=False)


### 3b) Headline classification metrics


In [ ]:
def _min_positives(n_patients, threshold=EVAL_PREVALENCE_THRESHOLD):
    return max(1, int(round(threshold * n_patients)))


def per_patient_auc(predictions, gt_episodes, outcome_names, min_positives=None):
    """
    Each patient contributes a single (risk_score, label) pair per outcome.
    risk_score = predictions.loc[pid, "P_<outcome>"]; label = 1 iff the outcome
    occurred in the un-truncated GT trajectory.
    """
    n_patients = predictions.shape[0]
    if min_positives is None:
        min_positives = _min_positives(n_patients)

    rows = []
    for name in outcome_names:
        pcol = f"P_{name}"
        if pcol not in predictions.columns:
            continue
        scores = predictions[pcol].to_numpy()
        labels = np.array([
            int(len(gt_episodes.get(pid, {}).get(name, [])) > 0)
            for pid in predictions.index
        ])
        n_pos = int(labels.sum()); n_neg = int(len(labels) - n_pos)
        prevalence = n_pos / max(1, n_pos + n_neg)
        if n_pos < min_positives or n_neg < min_positives:
            rows.append({"outcome": name, "auroc": np.nan, "auprc": np.nan,
                         "max_f1": np.nan, "f1_at_0_5": np.nan,
                         "n_pos": n_pos, "n_neg": n_neg, "prevalence": prevalence})
            continue
        precisions, recalls, thresholds = precision_recall_curve(labels, scores)
        f1s = np.where((precisions + recalls) > 0,
                       2 * precisions * recalls / np.maximum(precisions + recalls, 1e-12), 0.0)
        best_idx = int(np.argmax(f1s))
        max_f1 = float(f1s[best_idx])
        preds_05 = (scores >= 0.5).astype(int)
        tp = int(((preds_05 == 1) & (labels == 1)).sum())
        fp = int(((preds_05 == 1) & (labels == 0)).sum())
        fn = int(((preds_05 == 0) & (labels == 1)).sum())
        prec_05 = tp / max(tp + fp, 1); rec_05 = tp / max(tp + fn, 1)
        f1_at_0_5 = 2 * prec_05 * rec_05 / (prec_05 + rec_05) if (prec_05 + rec_05) > 0 else 0.0
        rows.append({
            "outcome": name,
            "auroc": float(roc_auc_score(labels, scores)),
            "auprc": float(average_precision_score(labels, scores)),
            "max_f1": max_f1, "f1_at_0_5": float(f1_at_0_5),
            "n_pos": n_pos, "n_neg": n_neg, "prevalence": prevalence,
        })
    return pd.DataFrame(rows).set_index("outcome").sort_values("auroc", ascending=False)


def weighted_mean_auc(auc_table, by="n_pos"):
    tbl = auc_table.dropna(subset=["auroc"])
    if len(tbl) == 0:
        nan = float("nan")
        return {"auroc_weighted": nan, "auprc_weighted": nan,
                "max_f1_weighted": nan, "f1_at_0_5_weighted": nan,
                "auroc_simple": nan, "auprc_simple": nan,
                "max_f1_simple": nan, "f1_at_0_5_simple": nan, "n_outcomes_used": 0}
    w = tbl[by].astype(float).values
    w = w / w.sum() if w.sum() > 0 else np.ones_like(w) / len(w)
    return {
        "auroc_weighted":     float((tbl["auroc"].values     * w).sum()),
        "auprc_weighted":     float((tbl["auprc"].values     * w).sum()),
        "max_f1_weighted":    float((tbl["max_f1"].values    * w).sum()),
        "f1_at_0_5_weighted": float((tbl["f1_at_0_5"].values * w).sum()),
        "auroc_simple":   float(tbl["auroc"].mean()),
        "auprc_simple":   float(tbl["auprc"].mean()),
        "max_f1_simple":  float(tbl["max_f1"].mean()),
        "f1_at_0_5_simple": float(tbl["f1_at_0_5"].mean()),
        "n_outcomes_used": int(len(tbl)),
    }


auc_outcome_names = [n for n in OUTCOME_NAMES if n not in AUC_EXCLUDE]
auc_table = per_patient_auc(predictions, gt_episodes, auc_outcome_names)
headline  = weighted_mean_auc(auc_table, by="n_pos")
print("\n[Patient AUC headline]")
for k in ("auroc_weighted", "auprc_weighted", "max_f1_weighted", "f1_at_0_5_weighted",
          "n_outcomes_used"):
    print(f"  {k:<22s} = {headline[k]:.4f}" if isinstance(headline[k], float) else
          f"  {k:<22s} = {headline[k]}")
auc_table.round(4)


### 3c) Length-of-stay regression (RELEASE time head)


In [ ]:
def length_of_stay_mae(predictions, gt_episodes, release_token=RELEASE_TOKEN):
    """
    LoS = predicted hours from sequence start to RELEASE — read from the
    time head's RELEASE slot.  GT LoS = first GT RELEASE_EVENT TimePoint.
    """
    tcol = f"T_{release_token}"
    if tcol not in predictions.columns:
        return {"mae_hours": float("nan"), "n_patients": 0}
    errs, gts, preds = [], [], []
    for pid in predictions.index:
        gt_releases = gt_episodes.get(pid, {}).get(release_token, [])
        if not gt_releases:
            continue
        gt_los   = float(min(gt_releases))
        pred_los = float(predictions.loc[pid, tcol])
        errs.append(abs(pred_los - gt_los))
        gts.append(gt_los); preds.append(pred_los)
    if not errs:
        return {"mae_hours": float("nan"), "n_patients": 0}
    arr = np.asarray(errs)
    return {
        "mae_hours":    float(arr.mean()),
        "median_hours": float(np.median(arr)),
        "p90_hours":    float(np.percentile(arr, 90)),
        "n_patients":   int(len(arr)),
        "gt_mean_hours":   float(np.mean(gts)),
        "pred_mean_hours": float(np.mean(preds)),
    }


los = length_of_stay_mae(predictions, gt_episodes)
print(f"\nLength-of-stay MAE: {los['mae_hours']:.1f}h  "
      f"(median {los.get('median_hours', float('nan')):.1f}h, "
      f"p90 {los.get('p90_hours', float('nan')):.1f}h, n={los['n_patients']})")
print(f"  GT mean = {los.get('gt_mean_hours', float('nan')):.1f}h  "
      f"pred mean = {los.get('pred_mean_hours', float('nan')):.1f}h")


### 3d) Time-head MAE per outcome (positives only)


In [ ]:
def time_head_mae(predictions, gt_episodes, outcome_names):
    """
    Mean absolute error between the time head's per-outcome prediction (hours
    from sequence start) and the *nearest* GT occurrence among patients where
    the outcome actually appears.
    """
    rows = []
    for name in outcome_names:
        tcol = f"T_{name}"
        if tcol not in predictions.columns:
            continue
        errs = []
        for pid in predictions.index:
            episodes = gt_episodes.get(pid, {}).get(name, [])
            if not episodes:
                continue
            pred_t = float(predictions.loc[pid, tcol])
            err = min(abs(pred_t - t_gt) for t_gt in episodes)
            errs.append(err)
        rows.append({"outcome": name,
                     "mae_hours": float(np.mean(errs)) if errs else float("nan"),
                     "n_patients": len(errs)})
    return pd.DataFrame(rows).set_index("outcome").sort_values("mae_hours")


time_mae_table = time_head_mae(predictions, gt_episodes, OUTCOME_NAMES)
time_mae_table.round(2)


### 3e) Temperature calibration

Per-outcome temperature scaling on the risk-head logits.  Does not change rank
order; improves reliability of the reported probabilities.


In [ ]:
@torch.no_grad()
def _collect_risk_logits(model, dataset_):
    """Run inference once and return raw risk logits aligned to outcome_names."""
    from torch.utils.data import DataLoader
    device = next(model.parameters()).device
    loader = DataLoader(dataset_, batch_size=model_config.TRAINING_SETTINGS["batch_size"],
                        shuffle=False, collate_fn=dataset.collate_emr, num_workers=0)
    chunks = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        risk_logits, _, _, _ = model.predict(
            parent_raw_ids=batch["parent_raw_ids"],
            concept_ids=batch["concept_ids"],
            value_ids=batch["value_ids"],
            position_ids=batch["position_ids"],
            abs_ts=batch["abs_ts"],
            context_vec=batch["context_vec"],
        )
        chunks.append(risk_logits.cpu())
    return torch.cat(chunks, dim=0)


def calibrate_temperature(model, dataset_, gt_first, outcome_names,
                           n_iter=200, lr=0.05):
    risk_idx = model.task_heads.risk_idx.cpu().tolist()
    risk_names = [outcome_names[i] for i in risk_idx]
    logits = _collect_risk_logits(model, dataset_)              # [N, K_risk]
    all_pids = list(dataset_.patient_ids)
    temperatures, cal_data = {}, {}
    for j, name in enumerate(risk_names):
        labels = np.array([int(gt_first.get(pid, {}).get(name, np.inf) < np.inf)
                           for pid in all_pids], dtype=np.float32)
        log_j = logits[:, j].float()
        if labels.sum() < 2:
            temperatures[name] = 1.0
            cal_data[name] = (log_j.numpy(), labels)
            continue
        T = torch.nn.Parameter(torch.ones(1))
        opt = torch.optim.LBFGS([T], lr=lr, max_iter=n_iter)
        labels_t = torch.tensor(labels)
        def closure():
            opt.zero_grad()
            loss = torch.nn.functional.binary_cross_entropy_with_logits(log_j / T, labels_t)
            loss.backward()
            return loss
        opt.step(closure)
        temperatures[name] = float(T.item())
        cal_data[name] = (log_j.numpy(), labels)
    return temperatures, cal_data


temperatures, cal_data = calibrate_temperature(encoder, eval_ds_input, gt_first, OUTCOME_NAMES)
pd.Series(temperatures, name="temperature_T").sort_values()


### 3f) Reliability diagrams


In [ ]:
def reliability_diagram(ax, logits, labels, T=1.0, n_bins=10, label=""):
    probs = torch.sigmoid(torch.tensor(logits) / T).numpy()
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers, mean_true = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() > 0:
            bin_centers.append(probs[mask].mean())
            mean_true.append(labels[mask].mean())
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect")
    ax.plot(bin_centers, mean_true, "o-", label=label)
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Fraction of positives")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(fontsize=8)


eligible = [n for n in cal_data if cal_data[n][1].sum() >= 5]
n_plots = min(len(eligible), 8)
if n_plots:
    fig, axes = plt.subplots(2, (n_plots + 1) // 2, figsize=(4 * ((n_plots + 1) // 2), 8))
    axes = np.atleast_1d(axes).flatten()
    for ax, name in zip(axes, eligible[:n_plots]):
        logits_np, labels_np = cal_data[name]
        ax.set_title(name.replace("_EVENT", "").replace("_COMPLICATION", ""), fontsize=8)
        reliability_diagram(ax, logits_np, labels_np, T=1.0,              label="Uncalibrated")
        reliability_diagram(ax, logits_np, labels_np, T=temperatures[name], label="Calibrated")
    for ax in axes[n_plots:]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.suptitle("Reliability Diagrams (risk head)", y=1.01, fontsize=11)
    plt.show()
else:
    print("Not enough positives for reliability diagrams.")


## 4) Notes on the BERT-pivot

* No autoregressive trajectory is generated — every patient is scored with a
  single bidirectional encoder pass through `inference.predict`.
* Headline metric is patient-level AUROC / AUPRC / F1 against the risk head;
  rank stability is identical to the AR `per_patient_max_auc` framework with
  the trajectory peak replaced by the per-patient probability.
* Length-of-stay regression is now read directly from the time head's
  RELEASE slot (`T_RELEASE_EVENT`) instead of the AR trajectory length.
* See `DILEMMAS.md` for design choices that are worth a morning conversation.
